In [2]:
import numpy as np
import torch_cluster
from GVP_esm import _rbf,_normalize
import torch_geometric

In [3]:
import torch

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "8"  

device = "cuda" if torch.cuda.is_available() else "cpu"

In [4]:

from GVP_esm import ProteinGraphDataset
class Protein_new(ProteinGraphDataset):
    def __init__(self, data_list, 
                     num_positional_embeddings=16,
                     top_k=30, num_rbf=16, device="cpu"):
            super(Protein_new, self).__init__(data_list, num_positional_embeddings, top_k, num_rbf, device)
    def _featurize_as_graph(self, protein):
        name = protein['name']
        with torch.no_grad():
            # protein['coords'] = list(zip(
            #     protein['coords']['N'], protein['coords']['CA'], protein['coords']['C'], protein['coords']['O']
            # ))
                
            coords = torch.as_tensor(protein['coords'], 
                                     device=self.device, dtype=torch.float32)   
            # seq = torch.as_tensor([self.letter_to_num[a] for a in protein['seq']],
            #                       device=self.device, dtype=torch.long)
            seq = protein['seq']
            # embedding = torch.as_tensor(protein['embedding'], 
            #                          device=self.device, dtype=torch.float32)   
            mask = torch.isfinite(coords.sum(dim=(1,2)))
            coords[~mask] = np.inf
            
            X_ca = coords[:, 1]
            edge_index = torch_cluster.knn_graph(X_ca, k=self.top_k)
            
            pos_embeddings = self._positional_embeddings(edge_index)
            E_vectors = X_ca[edge_index[0]] - X_ca[edge_index[1]]
            rbf = _rbf(E_vectors.norm(dim=-1), D_count=self.num_rbf, device=self.device)
            
            dihedrals = self._dihedrals(coords)                     
            orientations = self._orientations(X_ca)
            sidechains = self._sidechains(coords)
            
            node_s = dihedrals
            node_v = torch.cat([orientations, sidechains.unsqueeze(-2)], dim=-2)
            edge_s = torch.cat([rbf, pos_embeddings], dim=-1)
            edge_v = _normalize(E_vectors).unsqueeze(-2)
            
            node_s, node_v, edge_s, edge_v = map(torch.nan_to_num,
                    (node_s, node_v, edge_s, edge_v))
        data = torch_geometric.data.Data(x=X_ca, seq=seq, name=name,
                                         node_s=node_s, node_v=node_v,
                                         edge_s=edge_s, edge_v=edge_v,
                                         edge_index=edge_index, mask=mask,
                                        )
        return data

In [5]:
from Bio import PDB
def get_coo(pdb_path):
    parser = PDB.PDBParser()
    structure = parser.get_structure('protein', pdb_path)

    # 定义主链原子
    main_chain_atoms = ['N', 'CA', 'C', 'O']

    # 提取主链原子坐标
    main_chain_coords = []

    for model in structure:
        for chain in model:
            for residue in chain:
                coo = []
                for atom in residue:
                    if atom.get_name() in main_chain_atoms:
                        coord = atom.get_coord()
                        coo.append(coord)
                main_chain_coords.append(coo)
    return main_chain_coords

In [6]:
from GVP_esm import GVP_ESM3

In [7]:
esm_model_name = "/home/Wangtl2022/data1/ESM/esm_t33"
# original_model_weights_path = "../GVP_ESM_con_bk4.pth"
# 0:drop_rate = 0.1,num_layers = 5,emb_red = 192
# 1:drop_rate = 0.2,num_layers = 1,emb_red = 192
# 2:drop_rate = 0.2,num_layers = 2,emb_red = 192
# 3:drop_rate = 0.1,num_layers = 2,emb_red = 64
# 4:drop_rate = 0.0,num_layers = 3,emb_red = 128

model0 = GVP_ESM3(esm_model_name, "../GVP_ESM_con_bk0.pth",device,drop_rate = 0.1,num_layers = 5,emb_red = 192)
model1 = GVP_ESM3(esm_model_name, "../GVP_ESM_con_bk1.pth",device,drop_rate = 0.2,num_layers = 1,emb_red = 192)
model2 = GVP_ESM3(esm_model_name, "../GVP_ESM_con_bk2.pth",device,drop_rate = 0.2,num_layers = 2,emb_red = 192)
model3 = GVP_ESM3(esm_model_name, "../GVP_ESM_con_bk3.pth",device,drop_rate = 0.1,num_layers = 2,emb_red = 64)
model4 = GVP_ESM3(esm_model_name, "../GVP_ESM_con_bk4.pth",device,drop_rate = 0.0,num_layers = 3,emb_red = 128)

Some weights of EsmModel were not initialized from the model checkpoint at /home/Wangtl2022/data1/ESM/esm_t33 and are newly initialized: ['esm.pooler.dense.bias', 'esm.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of EsmModel were not initialized from the model checkpoint at /home/Wangtl2022/data1/ESM/esm_t33 and are newly initialized: ['esm.pooler.dense.bias', 'esm.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of EsmModel were not initialized from the model checkpoint at /home/Wangtl2022/data1/ESM/esm_t33 and are newly initialized: ['esm.pooler.dense.bias', 'esm.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of EsmModel were not initialized from the model checkpoint at /home/Wangtl2022/data1

In [8]:
import pandas as pd

file_path = "./A1LCD.xlsx"
df = pd.read_excel(file_path, sheet_name='f1_f4')


entrys0 = []

a1lcd0_path = "/home/Wangtl2022/data1/A1LCD/A1LCD0/out/"
for i in range(len(df)):
    seq = df.iloc[i]['seq']
    name = df.iloc[i]['sequence_name']
    path = os.path.join(a1lcd0_path,name,'ranked_0.pdb')
    coo = get_coo(path)
    entrys0.append({"name":name,"seq":seq,"coords":coo})

In [9]:
labels0 = df['critical_concentration'].to_numpy()

In [11]:
def run(entrys,device0,model):
    dataset = Protein_new(entrys)
    model.to(device0)
    model.eval()

    results = []

    attns = []
        # def forward(self,sequences, h_V, edge_index, h_E, batch=None):     
    for idx,data in enumerate(dataset):  # Iterate in batches over the training dataset.
    #         out = model(data.x, data.edge_index, data.batch)  # Perform a single forward pass.
    #         print(idx)
        # data = data.to(device)

        h_V = (data.node_s.to(device0), data.node_v.to(device0))

        h_E = (data.edge_s.to(device0), data.edge_v.to(device0)) 

        with torch.no_grad():


            out,attn = model(sequences = data.seq, h_V = h_V, edge_index = data.edge_index.to(device0), h_E = h_E)
            results.append(out.detach().cpu())
            attns.append(attn.detach().cpu())
    return results,attns

In [12]:

results0, attns0 = run(entrys0,device,model0)
results1, attns1 = run(entrys0,device,model1)
results2, attns2 = run(entrys0,device,model2)
results3, attns3 = run(entrys0,device,model3)
results4, attns4 = run(entrys0,device,model4)

/data1/data_wtl/GVP/A1LCD/GVP_esm.py:622: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  alphas = self.softmax(penalized_alphas.view(-1, size[1]))


In [13]:
import pandas as pd

file_path = "./A1LCD.xlsx"
df = pd.read_excel(file_path, sheet_name='f7')


entrys1 = []

#结构文件路径
a1lcd1_path = "/home/Wangtl2022/data1/A1LCD/A1LCD1/out/"

for i in range(len(df)):
    seq = df.iloc[i]['seq']
    name = df.iloc[i]['sequence_name']
    path = os.path.join(a1lcd1_path,name,'ranked_0.pdb')
    coo = get_coo(path)
    entrys1.append({"name":name,"seq":seq,"coords":coo})

In [14]:
results01, attns01 = run(entrys1,device,model0)
results11, attns11 = run(entrys1,device,model1)
results21, attns21 = run(entrys1,device,model2)
results31, attns31 = run(entrys1,device,model3)
results41, attns41 = run(entrys1,device,model4)

/data1/data_wtl/GVP/A1LCD/GVP_esm.py:622: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  alphas = self.softmax(penalized_alphas.view(-1, size[1]))


In [18]:
results21

[tensor([1.1196]),
 tensor([1.0551]),
 tensor([1.0994]),
 tensor([1.0450]),
 tensor([1.0481]),
 tensor([0.9847]),
 tensor([1.0372]),
 tensor([1.0462]),
 tensor([1.0213]),
 tensor([1.0126]),
 tensor([0.9929]),
 tensor([0.9582]),
 tensor([0.8865])]

In [25]:
labels1 = df['critical_concentration ']
labels1 = labels1.to_numpy()

In [19]:
results0 = torch.cat(results0).numpy()
results1 = torch.cat(results1).numpy()
results2 = torch.cat(results2).numpy()
results3 = torch.cat(results3).numpy()
results4 = torch.cat(results4).numpy()
results01 = torch.cat(results01).numpy()
results11 = torch.cat(results11).numpy()
results21 = torch.cat(results21).numpy()
results31 = torch.cat(results31).numpy()
results41 = torch.cat(results41).numpy()

In [20]:
results_all0 = [results0,results1,results2,results3,results4]
results_all1 = [results01,results11,results21,results31,results41]

In [45]:
group0 = []
for i in results_all0:
    corr_coefficient, p_value = spearmanr(i, labels0)
    group0.append([corr_coefficient, p_value])

In [35]:
# mean

corr_coefficient, p_value = spearmanr(np.vstack(results_all0).mean(axis=0), labels0)
corr_coefficient, p_value 

(-0.590686274509804, 0.012536389668570057)